# 00. 선형대수 기초 (Linear Algebra Foundations)

이번 단원에서는 딥러닝에 필요한 **선형대수 기초**를 배워보겠습니다.

## 학습 목표
- 벡터와 행렬의 기하학적 의미
- 내적(Dot Product)과 행렬곱의 직관
- 선형 변환과 차원 변환의 이해
- 신경망 수학의 기초

## 왜 이 단원이 필요한가?

딥러닝의 모든 연산은 **행렬 연산**입니다. 신경망을 이해하려면 선형대수의 직관이 필수입니다.

**궁금증**: 신경망이 숫자를 어떻게 처리할까?
- **답**: 모든 데이터를 숫자 배열(벡터/행렬)로 표현하고 행렬 연산으로 변환

**예시**:
- 이미지 = 픽셀 행렬
- 텍스트 = 단어 벡터
- 주식 가격 = 시계열 벡터

**이 단원을 이해하면**:
- 텐서가 왜 필요한지 알 수 있음
- 행렬곱이 신경망의 핵심인 이유를 알 수 있음
- 이후 PyTorch 학습이 쉬워짐

**중요**: 여기서는 PyTorch를 몰라도 됩니다. 순수 수학 개념만 배웁니다!

## 이 단원을 배우기 전에

**고등학교 수학 복습**: 벡터와 행렬의 기본 연산을 기억하시나요?

## 이 단원 다음에는

**다음 단원 (00a)**: 선형대수를 이해했으니, 이제 미적분 기초를 배워봅시다.

---
# 1. 직관적 이해 (Why)
---

## 1.1 왜 선형대수가 필요한가?

### 딥러닝 = 행렬 연산

신경망의 모든 계산은 행렬 곱셈입니다!

**예시**: 이미지 분류
- 입력: 28×28 = 784개의 픽셀 (벡터)
- 가중치: 784×10 행렬
- 출력: 10개의 클래스 점수 (벡터)

**계산**: 벡터 × 행렬 = 벡터 (행렬 곱셈!)

### 기하학적 직관

- **벡터**: 데이터의 한 샘플 (숫자들의 배열)
- **행렬**: 변환 규칙 (공간을 바꾸는 함수)
- **행렬곱**: 데이터를 새로운 공간으로 투영
- **내적**: 두 벡터의 유사도 측정

**의미**: 신경망은 데이터를 "더 잘 분리되는 공간"으로 변환하는 행렬을 학습합니다!

### 이후 연결

나중에 PyTorch를 배우면, 이 행렬 곱셈이 바로 `nn.Linear` 레이어입니다!


---
# 2. 수학적 기초 (What)
---

## 2.1 벡터의 기하학적 의미

벡터는 **방향과 크기**를 가진 화살표입니다.

2차원 벡터: $\vec{v} = \begin{bmatrix} v_1 \\ v_2 \end{bmatrix}$

**기하학적 해석**:
- **크기**: $\|\vec{v}\| = \sqrt{v_1^2 + v_2^2}$
- **방향**: 원점에서 $(v_1, v_2)$로 가는 화살표

### 머신러닝에서의 의미

**입력 데이터**: 이미지, 텍스트, 사용자 특성 → 모두 벡터  
**가중치 벡터**: 각 뉴런이 학습하는 "패턴"


In [ ]:
# 벡터 시각화
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 1) 기본 벡터
ax1 = axes[0]
v = np.array([3, 4])
ax1.arrow(0, 0, v[0], v[1], head_width=0.3, head_length=0.3,
          fc='blue', ec='blue', linewidth=2, label='벡터 v = [3, 4]')
ax1.grid(True, alpha=0.3)
ax1.set_xlim(-1, 5)
ax1.set_ylim(-1, 5)
ax1.set_aspect('equal')
ax1.legend()
ax1.set_title('벡터의 기하학적 표현')

# 2) 여러 벡터 비교
ax2 = axes[1]
vectors = [np.array([3, 4]), np.array([-2, 3]), np.array([1, -2])]
colors = ['blue', 'red', 'green']
for v, color in zip(vectors, colors):
    ax2.arrow(0, 0, v[0], v[1], head_width=0.3, head_length=0.3,
              fc=color, ec=color, linewidth=2, label=f'v = [{v[0]}, {v[1]}]')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(-3, 4)
ax2.set_ylim(-3, 5)
ax2.set_aspect('equal')
ax2.legend()
ax2.set_title('여러 벡터 비교')

plt.tight_layout()
plt.show()

print("벡터의 성질:")
for v in vectors:
    print(f"- 벡터 {v}: 크기 = {np.linalg.norm(v):.2f}")
print("→ 머신러닝: 각 데이터 샘플이 하나의 벡터")


## 2.2 내적(Dot Product)의 의미

두 벡터의 내적:

$$\vec{a} \cdot \vec{b} = a_1 b_1 + a_2 b_2 = \|\vec{a}\| \|\vec{b}\| \cos(\theta)$$

**기하학적 의미**: 두 벡터의 **정렬도(alignment)** 측정
- 같은 방향 → 내적 최대
- 수직 → 내적 = 0
- 반대 방향 → 내적 최소

### 머신러닝 연결

**내적 = 유사도**: 두 벡터가 얼마나 비슷한 패턴인가?

**예시**: 고양이 이미지 인식
- 입력 벡터: 픽셀들의 배열
- 가중치 벡터: "고양이 패턴"
- 내적이 크다 → 이미지가 고양이와 유사!


In [ ]:
# 내적의 기하학적 의미 시각화
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

a = np.array([3, 0])
vectors_b = [
    (np.array([2, 0]), "같은 방향"),
    (np.array([0, 2]), "수직"),
    (np.array([-2, 0]), "반대 방향")
]

for i, (b, title) in enumerate(vectors_b):
    ax = axes[i]
    ax.arrow(0, 0, a[0], a[1], head_width=0.3, head_length=0.3,
             fc='blue', ec='blue', linewidth=2, label='벡터 a')
    ax.arrow(0, 0, b[0], b[1], head_width=0.3, head_length=0.3,
             fc='red', ec='red', linewidth=2, label='벡터 b')
    
    dot = np.dot(a, b)
    ax.set_title(f'{title}\n내적 = {dot:.1f}')
    ax.set_xlim(-3, 4)
    ax.set_ylim(-1, 3)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='k', linestyle='--', linewidth=0.5)
    ax.axvline(x=0, color='k', linestyle='--', linewidth=0.5)
    ax.legend()

plt.tight_layout()
plt.show()

print("내적의 의미:")
print("- 같은 방향: 내적 큼 → 유사한 패턴")
print("- 수직: 내적 0 → 무관한 패턴")
print("- 반대 방향: 내적 음수 → 반대 패턴")
print("→ 가중치 벡터와 입력의 내적이 크면 해당 뉴런 활성화!")


## 2.3 행렬의 기하학적 의미

행렬은 **공간을 변환**하는 함수입니다.

행렬곱: $A\vec{x} = \begin{bmatrix} a_{11} & a_{12} \\ a_{21} & a_{22} \end{bmatrix} \begin{bmatrix} x_1 \\ x_2 \end{bmatrix}$

**열벡터 관점**: 행렬의 열벡터들의 선형결합

$$A\vec{x} = x_1 \begin{bmatrix} a_{11} \\ a_{21} \end{bmatrix} + x_2 \begin{bmatrix} a_{12} \\ a_{22} \end{bmatrix}$$

### 머신러닝 연결

**행렬 곱셈 = 공간 변환**

각 출력 = 가중치 벡터와 입력의 내적

**의미**: 입력을 새로운 공간으로 투영하는 변환

**예시**: 10차원 입력을 5차원으로 변환
- 10×5 행렬로 곱하면 5차원 출력
- 각 출력은 "특정 패턴을 얼마나 가지고 있는가"


---
# 3. 이후 PyTorch에서 보면
---

## 3.1 텐서란?

**이미 배운 개념을 다른 이름으로 부르는 것입니다!**

PyTorch는 벡터와 행렬을 **텐서(tensor)**라는 이름으로 부릅니다:

- **스칼라**: 0차원 텐서 (하나의 숫자)
- **벡터**: 1차원 텐서 (숫자의 배열)
- **행렬**: 2차원 텐서 (벡터의 배열)
- **텐서**: 3차원 이상 (행렬의 배열의 배열...)

**중요**: 텐서는 새로운 개념이 아니라, 벡터와 행렬을 "다차원 배열"이라는 이름으로 일반화한 것입니다!


In [ ]:
# numpy로 벡터와 행렬 표현
from typing import Any
import numpy as np
from numpy._typing import NDArray

print("=== 차원별 배열 ===\n")

# 스칼라 (0차원)
scalar: NDArray[Any] = np.array(3.14)
print(f"스칼라: {scalar}, shape: {scalar.shape}, 차원: {scalar.ndim}D")

# 벡터 (1차원)
vector = np.array([1, 2, 3])
print(f"벡터: {vector}, shape: {vector.shape}, 차원: {vector.ndim}D")

# 행렬 (2차원)
matrix = np.array([[1, 2, 3], [4, 5, 6]])
print(f"행렬:\n{matrix}")
print(f"shape: {matrix.shape}, 차원: {matrix.ndim}D")

# 3차원 배열
array_3d = np.random.randn(2, 3, 4)
print(f"\n3D 배열 shape: {array_3d.shape}, 차원: {array_3d.ndim}D")
print("→ 2개의 행렬, 각 3×4 크기\n")

print("머신러닝 의미:")
print("- 이미지: (배치, 채널, 높이, 너비) → 4차원")
print("- 모든 데이터는 숫자 배열로 표현!")
print("\n참고: 나중에 PyTorch에서 이 배열을 '텐서'라고 부릅니다!")


## 3.2 신경망의 기본 연산

신경망의 가장 기본적인 계산은 **행렬곱 + 편향**입니다:

$$y = Wx + b$$

여기서:
- $x$: 입력 벡터
- $W$: 가중치 행렬 
- $b$: 편향 벡터
- $y$: 출력 벡터

**의미**: 
- 행렬 $W$가 입력 $x$를 새로운 공간으로 변환
- 편향 $b$가 기준점을 이동

**이후 연결**: 이 수식을 구현하는 것이 신경망 레이어입니다!


In [ ]:
# 행렬곱 + 편향 구현
import numpy as np

print("=== y = Wx + b 구현 ===\n")

np.random.seed(42)

# 입력 벡터 (4개 샘플, 각 5차원)
x = np.random.randn(4, 5)
print(f"입력 x shape: {x.shape}")

# 가중치 행렬 (3×5)
W = np.random.randn(3, 5)
print(f"가중치 W shape: {W.shape}")

# 편향 (3차원)
b = np.random.randn(3)
print(f"편향 b shape: {b.shape}\n")

# 행렬곱: W @ x.T (또는 x @ W.T)
y = x @ W.T + b
print(f"출력 y shape: {y.shape}")
print(f"\n출력 y:\n{y}\n")

print("핵심:")
print("- y = W @ x.T + b (행렬곱 + 편향)")
print("- 각 출력은 가중치 벡터와 입력의 내적!")
print("- 이것이 신경망 레이어의 기본 연산!")
print("\n참고: 나중에 PyTorch를 배우면 이걸 자동으로 해줍니다!")


---
# 4. 핵심 요약
---

## 이번 단원에서 배운 내용

### 1. 벡터와 행렬
- **벡터**: 방향과 크기를 가진 화살표 (데이터 샘플)
- **행렬**: 공간을 변환하는 함수 (선형 변환)
- **행렬곱**: 열벡터들의 선형결합으로 공간 변환
- **내적**: 두 벡터의 정렬도 측정 (유사도)

### 2. 머신러닝 연결
- **다차원 배열**: 벡터와 행렬을 일반화한 표현
- **기본 연산**: y = Wx + b (행렬곱 + 편향)
- **내적의 의미**: 두 벡터의 유사도 측정

### 3. 핵심 직관
- 모든 데이터는 숫자 배열(벡터/행렬)로 표현
- 행렬은 공간을 변환하는 함수
- 신경망은 데이터를 더 잘 분리되는 공간으로 변환

## 다음 단원 미리보기

**다음 단원 (00a)**: 미적분 기초
- 선형대수로 구조를 이해했으니, 이제 **미적분**을 배웁니다
- **미분**: 함수의 변화율 (기울기)
- **Gradient**: 손실을 줄이는 방향
- **Gradient Descent**: 학습 알고리즘의 수학적 원리

**이후**: Python/PyTorch
- 미적분을 배운 후 Python으로 구현하는 방법을 배웁니다!

수학 기초가 탄탄해졌으니, 이제 구현을 배울 준비가 되었습니다!
